# 01 — Graph generation and masks

This notebook is the entry point for understanding the data model used in the project.
It shows how to generate DAG transition matrices, inspect source/sink structure, and build
the control and target masks used in the optimization routines.

In [ ]:
import pathlib
import sys

ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

import torch
import matplotlib.pyplot as plt

from l2o_cyber_defense import (
    sample_dag,
    sample_dag_single_sink,
    make_dag_dataset,
    dag_sources_mask,
    dag_sinks_mask,
    control_mask_from_dag,
    dag_target_w,
)
from l2o_cyber_defense.visualization import build_graph_from_P, draw_graph_policy

## Sample one generic DAG

The matrix `P` is row-substochastic. In the cybersecurity interpretation, it can be seen as
a weighted attack-propagation or transition matrix over a directed graph.

In [ ]:
torch.manual_seed(0)

P = sample_dag(n=16, edge_prob=0.25, rho=0.9)
print("Shape:", tuple(P.shape))
print("Max row sum:", float(P.sum(dim=1).max()))
print("Min row sum:", float(P.sum(dim=1).min()))
print("Spectral radius upper bounded by row-sum max, so this is a stable prototype.")

In [ ]:
G = build_graph_from_P(P, edge_threshold=1e-3)
print("Number of nodes:", G.number_of_nodes())
print("Number of displayed edges:", G.number_of_edges())

## Sample a DAG with a unique sink

This construction is often convenient for debugging and for building easy-to-interpret toy examples.

In [ ]:
P_single = sample_dag_single_sink(n=20, edge_prob=0.30)

sources = dag_sources_mask(P_single)
sinks = dag_sinks_mask(P_single)
control_mask = control_mask_from_dag(P_single)
w = dag_target_w(P_single)

print("Number of sources:", int(sources.sum()))
print("Number of sinks:", int(sinks.sum()))
print("Number of controllable nodes:", int(control_mask.sum()))
print("Target vector sums to:", float(w.sum()))
print("Target support:", torch.nonzero(w > 0, as_tuple=False).flatten().tolist())

In [ ]:
# Use the target distribution as a dummy policy-like quantity only for visualization.
draw_graph_policy(
    P_single,
    pi=w,
    title="Single-sink DAG with target nodes highlighted",
    control_mask=control_mask,
)
plt.show()

## Generate a small dataset

Later notebooks will train on a dataset of DAGs rather than on a single instance.

In [ ]:
Ps, sizes = make_dag_dataset(num_mats=8, n_min=10, n_max=18, edge_prob=0.25, rho=0.9)
print("Dataset size:", len(Ps))
print("Instance sizes:", sizes)
print("Average size:", sum(sizes) / len(sizes))

## Suggested next step

Move to `02_fixed_point_policy_experiments.ipynb` to inspect the masked simplex policy and the fixed-point solver.